In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Set random seed for reproducibility
np.random.seed(42)

# Sample size
n = 500

# Generate respondent IDs
respondent_ids = [f"R{str(i).zfill(4)}" for i in range(1, n+1)]

# Demographics
ages = np.random.randint(18, 75, n)
sexes = np.random.choice(['Male', 'Female', 'Other'], n, p=[0.48, 0.50, 0.02])
regions = np.random.choice(['North', 'South', 'East', 'West', 'Central'], n,
                          p=[0.20, 0.22, 0.18, 0.20, 0.20])

# Pre-lockdown soft drink consumption (servings per week)
# Average 3-4 servings per week with variation
pre_lockdown_freq = np.random.gamma(3, 1.5, n)
pre_lockdown_freq = np.clip(pre_lockdown_freq, 0, 20).round(1)

# During lockdown consumption (generally increased due to stress, home availability)
# Some increased, some decreased, some stayed same
change_factor = np.random.choice(
    [0.5, 0.7, 0.9, 1.0, 1.2, 1.5, 2.0],
    n,
    p=[0.05, 0.10, 0.15, 0.20, 0.25, 0.20, 0.05]
)
during_lockdown_freq = (pre_lockdown_freq * change_factor).round(1)
during_lockdown_freq = np.clip(during_lockdown_freq, 0, 25)

# Self-reported health issues (more issues with higher consumption)
health_issues_pool = [
    'weight gain', 'fatigue', 'sleep problems', 'anxiety',
    'digestive issues', 'headaches', 'dental problems',
    'energy crashes', 'mood swings', 'high blood sugar'
]

def generate_health_issues(during_freq, pre_freq):
    """Generate health issues based on consumption levels"""
    consumption_increase = during_freq - pre_freq

    # Higher consumption = more likely to have issues
    if during_freq < 2:
        n_issues = np.random.choice([0, 1], p=[0.8, 0.2])
    elif during_freq < 5:
        n_issues = np.random.choice([0, 1, 2], p=[0.5, 0.4, 0.1])
    elif during_freq < 10:
        n_issues = np.random.choice([0, 1, 2, 3], p=[0.2, 0.4, 0.3, 0.1])
    else:
        n_issues = np.random.choice([1, 2, 3, 4], p=[0.2, 0.3, 0.3, 0.2])

    if n_issues == 0:
        return 'none'
    else:
        issues = np.random.choice(health_issues_pool, n_issues, replace=False)
        return ', '.join(sorted(issues))

health_issues = [generate_health_issues(d, p) for d, p in
                zip(during_lockdown_freq, pre_lockdown_freq)]

# Weight change (correlated with consumption increase)
# More consumption during lockdown = more weight gain
consumption_delta = during_lockdown_freq - pre_lockdown_freq
base_weight_change = consumption_delta * 0.3  # Rough correlation
weight_change_kg = base_weight_change + np.random.normal(0, 2, n)
weight_change_kg = np.clip(weight_change_kg, -10, 15).round(1)

# Notes (realistic survey comments)
notes_options = [
    'Stayed home more, easier access to soft drinks in fridge',
    'Stress from pandemic led to increased consumption',
    'Tried to cut back but found it difficult',
    'Working from home increased snacking and drinking',
    'Limited outdoor activities, relied more on comfort foods/drinks',
    'Started noticing health effects and reduced intake',
    'No change in habits',
    'Switched to diet versions to reduce sugar',
    'Bought in bulk during lockdown, consumed more',
    'Increased consumption during video calls and remote work',
    'Kids at home meant more soft drinks in the house',
    'Used as energy boost during long work hours',
    'Noticed weight gain and tried to reduce',
    'Replaced coffee with soft drinks',
    'Financial stress led to cheaper beverage choices',
    ''  # Some with no notes
]

notes = np.random.choice(notes_options, n,
                        p=[0.12, 0.12, 0.08, 0.12, 0.10, 0.06, 0.05,
                           0.07, 0.08, 0.06, 0.05, 0.04, 0.03, 0.01, 0.01, 0.00])

# Survey dates (collected during and shortly after lockdown)
start_date = datetime(2020, 6, 1)
dates = [start_date + timedelta(days=np.random.randint(0, 120)) for _ in range(n)]
dates = [d.strftime('%Y-%m-%d') for d in dates]

# Create DataFrame
df = pd.DataFrame({
    'respondent_id': respondent_ids,
    'age': ages,
    'sex': sexes,
    'region': regions,
    'pre_lockdown_freq_per_week': pre_lockdown_freq,
    'during_lockdown_freq_per_week': during_lockdown_freq,
    'self_reported_health_issues': health_issues,
    'weight_change_kg': weight_change_kg,
    'notes': notes,
    'date': dates
})

# Save to CSV
df.to_csv('consumer_survey_health.csv', index=False)

print("✓ Generated consumer_survey_health.csv")
print(f"\nDataset Overview:")
print(f"Total respondents: {len(df)}")
print(f"\nAge range: {df['age'].min()} - {df['age'].max()}")
print(f"\nConsumption Statistics:")
print(f"Pre-lockdown avg: {df['pre_lockdown_freq_per_week'].mean():.1f} servings/week")
print(f"During-lockdown avg: {df['during_lockdown_freq_per_week'].mean():.1f} servings/week")
print(f"Average change: {(df['during_lockdown_freq_per_week'] - df['pre_lockdown_freq_per_week']).mean():.1f} servings/week")
print(f"\nWeight change avg: {df['weight_change_kg'].mean():.1f} kg")
print(f"\nHealth Issues Distribution:")
print(df['self_reported_health_issues'].value_counts().head(10))
print(f"\nFirst 5 rows:")
print(df.head())

# Additional analysis
increased = (df['during_lockdown_freq_per_week'] > df['pre_lockdown_freq_per_week']).sum()
decreased = (df['during_lockdown_freq_per_week'] < df['pre_lockdown_freq_per_week']).sum()
same = (df['during_lockdown_freq_per_week'] == df['pre_lockdown_freq_per_week']).sum()

print(f"\n\nConsumption Changes:")
print(f"Increased: {increased} ({increased/n*100:.1f}%)")
print(f"Decreased: {decreased} ({decreased/n*100:.1f}%)")
print(f"Stayed same: {same} ({same/n*100:.1f}%)")

✓ Generated consumer_survey_health.csv

Dataset Overview:
Total respondents: 500

Age range: 18 - 74

Consumption Statistics:
Pre-lockdown avg: 4.7 servings/week
During-lockdown avg: 5.4 servings/week
Average change: 0.7 servings/week

Weight change avg: 0.2 kg

Health Issues Distribution:
self_reported_health_issues
none                199
anxiety              21
sleep problems       19
weight gain          17
mood swings          17
high blood sugar     15
headaches            15
dental problems      14
fatigue              12
digestive issues     12
Name: count, dtype: int64

First 5 rows:
  respondent_id  age     sex   region  pre_lockdown_freq_per_week  \
0         R0001   56  Female  Central                         8.5   
1         R0002   69    Male     East                         6.7   
2         R0003   46  Female     West                         4.1   
3         R0004   32    Male  Central                         2.6   
4         R0005   60    Male     West                  